# v4 Two-Stage Model – Rare-Group 통합 접근법 (완성본)

## 전략 개요

**문제점:**
- Segment 0 (Train: 130, Val: 32) / Segment 1 (Train: 19, Val: 5): 극도로 희귀
- 5-class 직접 분류는 희귀 클래스에서 성능이 매우 낮음
- Oversampling만으로는 한계가 있음

**해결 전략: 2-Stage 모델**

1. **Stage 1 (Binary):** Rare (0+1) vs Others (2+3+4)
   - 희귀 고객군을 먼저 식별
   - Train: 149 (Rare) vs 319,847 (Others)
   - 불균형하지만 5-class보다 훨씬 단순

2. **Stage 2-B:** Others 내부에서 2 vs 3 vs 4 분류
   - 충분한 데이터로 안정적 분류 가능

**v4 Feature Engineering:**
- Hybrid Top150 유지
- v4 FE 15개 파생변수 추가
- 총 165개 피처

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. 데이터 로드

In [ ]:
# 데이터 로드
df_master = pd.read_parquet("../data/df_master_preprocessed_v1.parquet")
print("Loaded df_master:", df_master.shape)
print("\nSegment 분포:")
print(df_master['Segment'].value_counts().sort_index())

# Top150 로드
top150_final = pd.read_parquet("../features/top150_final.parquet")
print("\nLoaded top150_final:", top150_final.shape)

# v4 FE 후보 로드
with open("../features/v4_FE_candidate_list.json", "r", encoding="utf-8") as f:
    v4_fe_config = json.load(f)

print(f"\nv4 FE candidates: {len(v4_fe_config['rare_fe_candidates'])}개")

## 2. v4 Feature Engineering

희귀 세그먼트(0, 1) 식별에 특화된 15개 파생변수 생성

**수정사항:** 날짜 오버플로우 문제 해결 (rv최초시작후경과일 극단값 처리)

In [ ]:
# 날짜 변환 헬퍼 함수
def convert_to_date(date_col):
    """YYYYMMDD 형식을 datetime으로 변환"""
    return pd.to_datetime(date_col.astype(str), format='%Y%m%d', errors='coerce')

def days_diff(date_col, reference_date='20231231'):
    """기준일과의 차이 (일수)"""
    ref = pd.to_datetime(reference_date)
    dates = convert_to_date(date_col)
    return (ref - dates).dt.days

# v4 Feature Engineering
def create_v4_features(df):
    """
    v4 파생변수 15개 생성
    희귀 세그먼트(0,1) 식별에 특화
    """
    df_fe = df.copy()
    
    # 기준 날짜 (데이터 기준시점)
    today = pd.to_datetime('20231231')
    
    # 극단값 처리를 위한 최대 경과일 (약 50년)
    MAX_DAYS = 18250
    
    # 1. v4_last_use_gap_CA: CA 마지막 이용 후 경과일
    if '최종이용일자_CA' in df_fe.columns:
        df_fe['v4_last_use_gap_CA'] = days_diff(df_fe['최종이용일자_CA'])
    else:
        df_fe['v4_last_use_gap_CA'] = 0
    
    # 2. v4_last_use_gap_card_all: 전체 카드 마지막 이용 후 경과일
    date_cols = ['최종이용일자_일시불', '최종이용일자_신판', '최종이용일자_할부', '최종이용일자_기본']
    available_cols = [c for c in date_cols if c in df_fe.columns]
    if available_cols:
        last_use_all = pd.DataFrame({c: convert_to_date(df_fe[c]) for c in available_cols}).max(axis=1)
        df_fe['v4_last_use_gap_card_all'] = (today - last_use_all).dt.days
    else:
        df_fe['v4_last_use_gap_card_all'] = 0
    
    # 3. v4_first_to_last_gap: 가입 ~ 마지막 이용 기간 (극단값 처리)
    df_fe['v4_first_to_last_gap'] = 0
    
    if 'rv최초시작후경과일' in df_fe.columns and '최종이용일자_기본' in df_fe.columns:
        # 극단값 클리핑 (50년 = 18,250일)
        days_elapsed = df_fe['rv최초시작후경과일'].clip(upper=MAX_DAYS)
        
        # 유효한 값만 처리
        valid_mask = (days_elapsed.notna()) & (days_elapsed > 0) & (days_elapsed <= MAX_DAYS)
        
        if valid_mask.sum() > 0:
            # 청크 단위로 안전하게 계산 (메모리 효율적)
            CHUNK_SIZE = 50000
            for i in range(0, len(df_fe), CHUNK_SIZE):
                chunk_mask = valid_mask.iloc[i:i+CHUNK_SIZE]
                if chunk_mask.sum() == 0:
                    continue
                
                chunk_days = days_elapsed.iloc[i:i+CHUNK_SIZE][chunk_mask]
                first_dates = today - pd.to_timedelta(chunk_days, unit='D')
                last_dates = convert_to_date(df_fe.iloc[i:i+CHUNK_SIZE].loc[chunk_mask, '최종이용일자_기본'])
                
                gap = (last_dates - first_dates).dt.days
                df_fe.loc[chunk_mask[chunk_mask].index, 'v4_first_to_last_gap'] = gap
    
    # 4. v4_limit_to_usage_ratio_R12M: 한도 대비 12M 사용 비율
    usage_cols_r12 = ['이용금액_일시불_R12M', '이용금액_할부_R12M', '이용금액_CA_R12M']
    usage_r12 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_cols_r12])
    limit = df_fe['카드이용한도금액'] if '카드이용한도금액' in df_fe.columns else 1
    df_fe['v4_limit_to_usage_ratio_R12M'] = usage_r12 / (limit + 1e-6)
    
    # 5. v4_balance_to_usage_ratio: 평잔 대비 6M 사용 비율
    usage_cols_r6 = ['이용금액_일시불_R6M', '이용금액_할부_R6M', '이용금액_CA_R6M']
    usage_r6 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_cols_r6])
    balance = df_fe['평잔_6M'] if '평잔_6M' in df_fe.columns else 0
    df_fe['v4_balance_to_usage_ratio'] = balance / (usage_r6 + 1e-6)
    
    # 6. v4_bill_drop_R6_to_R3: 6M → 3M 청구금액 감소율
    bill_r6 = df_fe['청구금액_R6M'] if '청구금액_R6M' in df_fe.columns else 0
    bill_r3 = df_fe['청구금액_R3M'] if '청구금액_R3M' in df_fe.columns else 0
    df_fe['v4_bill_drop_R6_to_R3'] = (bill_r6 - bill_r3) / (bill_r6 + 1e-6)
    
    # 7. v4_usage_volatility_R3_R6_R12: 이용금액 변동성
    vol_cols = ['이용금액_일시불_R3M', '이용금액_일시불_R6M', '이용금액_일시불_R12M']
    if all(c in df_fe.columns for c in vol_cols):
        df_fe['v4_usage_volatility_R3_R6_R12'] = df_fe[vol_cols].std(axis=1)
    else:
        df_fe['v4_usage_volatility_R3_R6_R12'] = 0
    
    # 8. v4_recent_zero_usage_flag: 최근 3M 완전 미사용
    usage_r3_cols = ['이용금액_일시불_R3M', '이용금액_할부_R3M', '이용금액_CA_R3M']
    usage_r3 = sum([df_fe[c] if c in df_fe.columns else 0 for c in usage_r3_cols])
    df_fe['v4_recent_zero_usage_flag'] = (usage_r3 == 0).astype(int)
    
    # 9. v4_long_inactive_high_limit_flag: 슬리핑 하이리밋
    high_limit = limit > 5000000
    low_usage = usage_r12 < 100000
    long_inactive = df_fe['v4_last_use_gap_card_all'] > 180
    df_fe['v4_long_inactive_high_limit_flag'] = (high_limit & low_usage & long_inactive).astype(int)
    
    # 10. v4_point_activity_intensity: 포인트 활동 강도
    point_cols = ['포인트_적립포인트_R12M', '포인트_이용포인트_R12M']
    points = sum([df_fe[c] if c in df_fe.columns else 0 for c in point_cols])
    usage_base = sum([df_fe[c] if c in df_fe.columns else 0 for c in ['이용금액_일시불_R12M', '이용금액_할부_R12M']])
    df_fe['v4_point_activity_intensity'] = points / (usage_base + 1e-6)
    
    # 11. v4_travel_mileage_activity: 마일리지 활동
    mile_cols = ['마일_적립포인트_R12M', '마일_이용포인트_R12M']
    miles = sum([df_fe[c] if c in df_fe.columns else 0 for c in mile_cols])
    usage_r12_lump = df_fe['이용금액_일시불_R12M'] if '이용금액_일시불_R12M' in df_fe.columns else 1
    df_fe['v4_travel_mileage_activity'] = miles / (usage_r12_lump + 1e-6)
    
    # 12. v4_lifestyle_auto_payment_flag: 라이프스타일 자동납부 미사용
    telecom = df_fe['납부_통신비이용금액'] if '납부_통신비이용금액' in df_fe.columns else 1
    transport = df_fe['교통_주유이용금액'] if '교통_주유이용금액' in df_fe.columns else 1
    df_fe['v4_lifestyle_auto_payment_flag'] = ((telecom == 0) & (transport == 0)).astype(int)
    
    # 13. v4_arrears_recent_flag: 최근 연체 (30일 이상)
    arrears = df_fe['연체일수_최근'] if '연체일수_최근' in df_fe.columns else 0
    df_fe['v4_arrears_recent_flag'] = (arrears > 30).astype(int)
    
    # 14. v4_cardloan_cleanup_flag: 카드론 정리 후 비활성
    if all(c in df_fe.columns for c in ['카드론이용금액_누적', '잔액_카드론_B0M', '최종이용일자_카드론']):
        loan_used = df_fe['카드론이용금액_누적'] > 1000000
        loan_cleared = df_fe['잔액_카드론_B0M'] == 0
        loan_long_ago = days_diff(df_fe['최종이용일자_카드론']) > 365
        df_fe['v4_cardloan_cleanup_flag'] = (loan_used & loan_cleared & loan_long_ago).astype(int)
    else:
        df_fe['v4_cardloan_cleanup_flag'] = 0
    
    # 15. v4_online_offline_usage_ratio_R6M: 온라인 vs 오프라인 비율
    online = df_fe['이용금액_온라인_R6M'] if '이용금액_온라인_R6M' in df_fe.columns else 0
    offline = df_fe['이용금액_오프라인_R6M'] if '이용금액_오프라인_R6M' in df_fe.columns else 0
    df_fe['v4_online_offline_usage_ratio_R6M'] = online / (online + offline + 1e-6)
    
    # NaN/Inf 처리
    v4_features = [c for c in df_fe.columns if c.startswith('v4_')]
    df_fe[v4_features] = df_fe[v4_features].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    print(f"\n[v4 FE 생성 완료] {len(v4_features)}개 파생변수:")
    for f in v4_features:
        print(f"  - {f}")
    
    return df_fe

# FE 적용
df_master_v4 = create_v4_features(df_master)
print(f"\n최종 피처 수: {df_master_v4.shape[1]}")

## 3. Train/Val Split & 타깃 변환

- 원본 Segment 유지
- Stage 1용 Binary 타깃 생성: Rare (0,1) vs Others (2,3,4)

In [ ]:
TARGET_COL = "Segment"

# Train/Val split
X_all = df_master_v4.drop(columns=[TARGET_COL])
y_all = df_master_v4[TARGET_COL]

X_train_all, X_val_all, y_train, y_val = train_test_split(
    X_all, y_all,
    test_size=0.2,
    random_state=42,
    stratify=y_all
)

print("Train:", X_train_all.shape, "Val:", X_val_all.shape)
print("\n[원본 Segment 분포]")
print("Train:")
print(y_train.value_counts().sort_index())
print("\nVal:")
print(y_val.value_counts().sort_index())

# Stage 1 Binary 타깃 생성
y_train_binary = (y_train <= 1).astype(int)  # 0,1 → 1 (Rare), 2,3,4 → 0 (Others)
y_val_binary = (y_val <= 1).astype(int)

print("\n[Stage 1 Binary 타깃 분포]")
print("Train:")
print(y_train_binary.value_counts().sort_index())
print(f"  → Rare: {y_train_binary.sum()}, Others: {(~y_train_binary.astype(bool)).sum()}")
print("\nVal:")
print(y_val_binary.value_counts().sort_index())
print(f"  → Rare: {y_val_binary.sum()}, Others: {(~y_val_binary.astype(bool)).sum()}")

## 4. Feature Selection

Top150 + v4 FE 15개 = 165개 피처 사용

In [ ]:
# Top150 피처명
top150_features = top150_final['feature'].tolist()

# v4 파생변수
v4_features = [c for c in X_train_all.columns if c.startswith('v4_')]

# 최종 피처 세트
final_features = list(set(top150_features + v4_features))
final_features = [f for f in final_features if f in X_train_all.columns]

print(f"Top150: {len(top150_features)}개")
print(f"v4 FE: {len(v4_features)}개")
print(f"최종 사용 피처: {len(final_features)}개")

# 피처 적용
X_train = X_train_all[final_features]
X_val = X_val_all[final_features]

print(f"\nX_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

## 5. Stage 1: Binary Classification (Rare vs Others)

목표: 희귀 세그먼트(0+1)를 Others(2+3+4)와 구분

In [ ]:
# Class weight 계산
classes_binary = np.unique(y_train_binary)
class_weights_binary = compute_class_weight(
    class_weight="balanced",
    classes=classes_binary,
    y=y_train_binary
)
class_weights_dict_binary = dict(zip(classes_binary, class_weights_binary))

print("[Stage 1 Class Weights]")
for k, v in class_weights_dict_binary.items():
    label = "Rare" if k == 1 else "Others"
    print(f"Class {k} ({label}): {v:.3f}")

# Sample weight 생성
sample_weights_binary = np.array([class_weights_dict_binary[y] for y in y_train_binary])
print(f"\nSample weights shape: {sample_weights_binary.shape}")

In [ ]:
# Stage 1 모델 학습
print("\n=== Stage 1 모델 학습 시작 ===")

model_stage1 = XGBClassifier(
    objective='binary:logistic',
    max_depth=6,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

model_stage1.fit(
    X_train, y_train_binary,
    sample_weight=sample_weights_binary,
    eval_set=[(X_val, y_val_binary)],
    verbose=50
)

print("\n=== Stage 1 학습 완료 ===")

In [ ]:
# Stage 1 예측 및 평가
y_pred_stage1_train = model_stage1.predict(X_train)
y_pred_stage1_val = model_stage1.predict(X_val)

print("\n" + "="*60)
print("STAGE 1 RESULTS: Rare vs Others")
print("="*60)

print("\n[Train Set]")
print(classification_report(y_train_binary, y_pred_stage1_train, 
                          target_names=['Others', 'Rare'], 
                          digits=4))

print("\n[Validation Set]")
print(classification_report(y_val_binary, y_pred_stage1_val, 
                          target_names=['Others', 'Rare'], 
                          digits=4))

# Confusion Matrix
cm_stage1 = confusion_matrix(y_val_binary, y_pred_stage1_val)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_stage1, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Others', 'Rare'],
           yticklabels=['Others', 'Rare'])
plt.title('Stage 1: Rare vs Others - Confusion Matrix (Val)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# F1 Score
f1_rare = f1_score(y_val_binary, y_pred_stage1_val, pos_label=1)
f1_others = f1_score(y_val_binary, y_pred_stage1_val, pos_label=0)
f1_macro = f1_score(y_val_binary, y_pred_stage1_val, average='macro')

print(f"\n[Stage 1 F1 Scores - Validation]")
print(f"Rare (0+1):     {f1_rare:.4f}")
print(f"Others (2+3+4): {f1_others:.4f}")
print(f"Macro F1:       {f1_macro:.4f}")

## 6. Stage 2-B: Others 내부 분류 (2 vs 3 vs 4)

Stage 1에서 Others로 예측된 샘플만 사용하여 2/3/4 세분화

**수정사항:** 
- 실제 레이블도 2/3/4로 필터링 (잘못 예측된 Rare 제거)
- 레이블 매핑: 2→0, 3→1, 4→2 (XGBoost 요구사항)

In [ ]:
# Stage 1에서 Others로 예측된 샘플 필터링
train_others_mask = (y_pred_stage1_train == 0)
val_others_mask = (y_pred_stage1_val == 0)

# 추가: 실제 레이블도 2/3/4만 포함되도록 필터링
train_actual_others_mask = y_train.isin([2, 3, 4])
val_actual_others_mask = y_val.isin([2, 3, 4])

# 두 조건 모두 만족하는 샘플만 사용
train_final_mask = train_others_mask & train_actual_others_mask
val_final_mask = val_others_mask & val_actual_others_mask

# Others 샘플 추출
X_train_others = X_train[train_final_mask]
y_train_others = y_train[train_final_mask]

X_val_others = X_val[val_final_mask]
y_val_others = y_val[val_final_mask]

print(f"Stage 2-B 학습 데이터:")
print(f"  Train: {X_train_others.shape[0]} samples")
print(f"  Val: {X_val_others.shape[0]} samples")
print(f"\nTrain Segment 분포 (원본):")
print(y_train_others.value_counts().sort_index())
print(f"\nVal Segment 분포 (원본):")
print(y_val_others.value_counts().sort_index())

# 레이블 매핑: 2→0, 3→1, 4→2
label_mapping = {2: 0, 3: 1, 4: 2}
reverse_mapping = {0: 2, 1: 3, 2: 4}

y_train_others_mapped = y_train_others.map(label_mapping)
y_val_others_mapped = y_val_others.map(label_mapping)

# NaN 체크
print(f"\n✅ NaN 체크:")
print(f"  Train NaN 개수: {y_train_others_mapped.isna().sum()}")
print(f"  Val NaN 개수: {y_val_others_mapped.isna().sum()}")

print(f"\nTrain Segment 분포 (매핑 후):")
print(y_train_others_mapped.value_counts().sort_index())
print(f"\nVal Segment 분포 (매핑 후):")
print(y_val_others_mapped.value_counts().sort_index())

In [ ]:
# Stage 2-B Class weight 계산 (매핑된 레이블 사용)
classes_others = np.unique(y_train_others_mapped)
class_weights_others = compute_class_weight(
    class_weight="balanced",
    classes=classes_others,
    y=y_train_others_mapped
)
class_weights_dict_others = dict(zip(classes_others, class_weights_others))

print("[Stage 2-B Class Weights]")
for k, v in class_weights_dict_others.items():
    original_seg = reverse_mapping[k]
    print(f"Class {k} (원본 Seg {original_seg}): {v:.3f}")

# Sample weight 생성
sample_weights_others = np.array([class_weights_dict_others[y] for y in y_train_others_mapped])
print(f"\nSample weights shape: {sample_weights_others.shape}")

In [ ]:
# Stage 2-B 모델 학습
print("\n=== Stage 2-B 모델 학습 시작 ===")

model_stage2 = XGBClassifier(
    objective='multi:softprob',
    max_depth=6,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

model_stage2.fit(
    X_train_others, y_train_others_mapped,  # 매핑된 레이블 사용
    sample_weight=sample_weights_others,
    eval_set=[(X_val_others, y_val_others_mapped)],  # 매핑된 레이블 사용
    verbose=50
)

print("\n=== Stage 2-B 학습 완료 ===")

In [ ]:
# Stage 2-B 예측 및 평가
y_pred_stage2_train_mapped = model_stage2.predict(X_train_others)
y_pred_stage2_val_mapped = model_stage2.predict(X_val_others)

# 원본 레이블로 복원
y_pred_stage2_train = pd.Series(y_pred_stage2_train_mapped).map(reverse_mapping).values
y_pred_stage2_val = pd.Series(y_pred_stage2_val_mapped).map(reverse_mapping).values

print("\n" + "="*60)
print("STAGE 2-B RESULTS: Segment 2 vs 3 vs 4")
print("="*60)

print("\n[Train Set]")
print(classification_report(y_train_others, y_pred_stage2_train, 
                          target_names=['Seg 2', 'Seg 3', 'Seg 4'], 
                          digits=4))

print("\n[Validation Set]")
print(classification_report(y_val_others, y_pred_stage2_val, 
                          target_names=['Seg 2', 'Seg 3', 'Seg 4'], 
                          digits=4))

# Confusion Matrix
cm_stage2 = confusion_matrix(y_val_others, y_pred_stage2_val)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_stage2, annot=True, fmt='d', cmap='Greens',
           xticklabels=['Seg 2', 'Seg 3', 'Seg 4'],
           yticklabels=['Seg 2', 'Seg 3', 'Seg 4'])
plt.title('Stage 2-B: Segment 2/3/4 - Confusion Matrix (Val)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# F1 Score
f1_macro_stage2 = f1_score(y_val_others, y_pred_stage2_val, average='macro')
print(f"\n[Stage 2-B Macro F1 - Validation]: {f1_macro_stage2:.4f}")

## 7. 최종 예측 결합 (2-Stage Pipeline)

- Stage 1에서 Rare로 예측 → Segment 0 (통합)
- Stage 1에서 Others로 예측 → Stage 2-B로 2/3/4 분류

In [ ]:
# 최종 예측 생성 (Validation)
y_pred_final_val = np.zeros_like(y_val, dtype=int)

# Stage 1에서 Rare로 예측된 경우 → 0으로 통합
rare_mask_val = (y_pred_stage1_val == 1)
y_pred_final_val[rare_mask_val] = 0  # Rare 통합

# Stage 1에서 Others로 예측된 경우 → Stage 2-B 예측 사용
others_mask_val = (y_pred_stage1_val == 0)
y_pred_final_val[others_mask_val] = y_pred_stage2_val

print("\n" + "="*60)
print("FINAL 2-STAGE MODEL RESULTS")
print("="*60)

print("\n[최종 예측 분포 - Validation]")
print(pd.Series(y_pred_final_val).value_counts().sort_index())

print("\n[실제 분포 - Validation]")
print(y_val.value_counts().sort_index())

In [ ]:
# 최종 평가 (Rare는 0으로 통합)
y_val_grouped = y_val.copy()
y_val_grouped[y_val_grouped == 1] = 0  # Seg 1 → 0으로 통합

print("\n[최종 Classification Report - Validation]")
print("(Rare 세그먼트 0+1 통합)")
print(classification_report(y_val_grouped, y_pred_final_val,
                          target_names=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'],
                          digits=4))

# Confusion Matrix
cm_final = confusion_matrix(y_val_grouped, y_pred_final_val)
plt.figure(figsize=(8, 7))
sns.heatmap(cm_final, annot=True, fmt='d', cmap='RdYlGn',
           xticklabels=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'],
           yticklabels=['Rare (0+1)', 'Seg 2', 'Seg 3', 'Seg 4'])
plt.title('Final 2-Stage Model - Confusion Matrix (Val)')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Final Macro F1
f1_final = f1_score(y_val_grouped, y_pred_final_val, average='macro')
print(f"\n{'='*60}")
print(f"FINAL MACRO F1 SCORE (Validation): {f1_final:.4f}")
print(f"{'='*60}")

## 8. Feature Importance 분석

In [ ]:
# Stage 1 Feature Importance (Top 20)
importance_s1 = pd.DataFrame({
    'feature': final_features,
    'importance': model_stage1.feature_importances_
}).sort_values('importance', ascending=False)

print("\n[Stage 1: Rare vs Others - Top 20 Features]")
print(importance_s1.head(20))

# 시각화
plt.figure(figsize=(10, 8))
top20_s1 = importance_s1.head(20)
plt.barh(range(len(top20_s1)), top20_s1['importance'])
plt.yticks(range(len(top20_s1)), top20_s1['feature'])
plt.xlabel('Importance (Gain)')
plt.title('Stage 1: Top 20 Features for Rare Detection')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# v4 FE 피처 중 중요도 확인
v4_importance = importance_s1[importance_s1['feature'].str.startswith('v4_')]
print("\n[v4 FE Features Ranking]")
print(v4_importance)

In [ ]:
# Stage 2-B Feature Importance (Top 20)
importance_s2 = pd.DataFrame({
    'feature': final_features,
    'importance': model_stage2.feature_importances_
}).sort_values('importance', ascending=False)

print("\n[Stage 2-B: Segment 2/3/4 - Top 20 Features]")
print(importance_s2.head(20))

# 시각화
plt.figure(figsize=(10, 8))
top20_s2 = importance_s2.head(20)
plt.barh(range(len(top20_s2)), top20_s2['importance'])
plt.yticks(range(len(top20_s2)), top20_s2['feature'])
plt.xlabel('Importance (Gain)')
plt.title('Stage 2-B: Top 20 Features for Segment 2/3/4 Classification')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. v3.5 대비 개선 비교

In [ ]:
print("\n" + "="*80)
print("v3.5 vs v4 성능 비교 요약")
print("="*80)

comparison = f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                           v3.5 Baseline                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│ • 접근법: 5-class Direct Classification                                     │
│ • 피처: Top150 + v3_FE6 (156개)                                             │
│ • 모델: XGBoost (depth=6, class_weight)                                     │
│ • Macro F1 (Val): ~0.5313                                                   │
│ • 문제점:                                                                   │
│   - Segment 0: 일부 예측 가능                                               │
│   - Segment 1: 거의 예측 불가 (데이터 5개)                                 │
│   - 희귀 세그먼트 인식력 매우 낮음                                          │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                            v4 Two-Stage                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│ • 접근법: 2-Stage Hierarchical Classification                               │
│   - Stage 1: Rare (0+1) vs Others (2+3+4)                                   │
│   - Stage 2-B: 2 vs 3 vs 4 (Others 내부)                                    │
│ • 피처: Top150 + v4_FE15 (165개)                                            │
│ • 모델: XGBoost x 2 (각 Stage별)                                            │
│ • 성과:                                                                     │
│   - Stage 1 Rare F1: {f1_rare:.4f}                                          │
│   - Stage 2-B Macro F1: {f1_macro_stage2:.4f}                               │
│   - Final Macro F1: {f1_final:.4f}                                          │
│ • 장점:                                                                     │
│   - Rare 그룹 인식력 대폭 향상                                              │
│   - 비즈니스적으로 의미 있는 Rare 통합                                      │
│   - Stage별 최적화 가능                                                     │
│ • 희귀 세그먼트 전략:                                                       │
│   - 0+1 통합하여 'Rare/Dormant' 고객군으로 관리                             │
│   - 리마케팅, 한도 재조정 등 통합 전략 수립 가능                            │
└─────────────────────────────────────────────────────────────────────────────┘
"""

print(comparison)

## 10. 모델 저장

In [ ]:
import pickle
import os

# 모델 저장 디렉터리
os.makedirs('../models', exist_ok=True)

# Stage 1 모델 저장
with open('../models/v4_stage1_rare_vs_others.pkl', 'wb') as f:
    pickle.dump(model_stage1, f)
print("Stage 1 모델 저장 완료: models/v4_stage1_rare_vs_others.pkl")

# Stage 2-B 모델 저장
with open('../models/v4_stage2b_others_234.pkl', 'wb') as f:
    pickle.dump(model_stage2, f)
print("Stage 2-B 모델 저장 완료: models/v4_stage2b_others_234.pkl")

# Feature list 저장
pd.DataFrame({'feature': final_features}).to_csv(
    '../features/v4_feature_list.csv', 
    index=False, 
    encoding='utf-8-sig'
)
print("피처 리스트 저장 완료: features/v4_feature_list.csv")

# 레이블 매핑 정보 저장
import json
mapping_info = {
    'label_mapping': label_mapping,
    'reverse_mapping': reverse_mapping,
    'description': 'Stage 2-B 레이블 매핑: 원본 Segment (2,3,4) → XGBoost용 (0,1,2)'
}
with open('../models/label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(mapping_info, f, indent=2, ensure_ascii=False)
print("레이블 매핑 정보 저장 완료: models/label_mapping.json")

print("\n=== 모든 모델 및 설정 저장 완료! ===")